# RegimeShift: Market-Regime Asset Allocation System
## IIT Bombay Summer Quant 2026 — Final Submission

This notebook reproduces the complete RegimeShift pipeline:
data → features → HMM → portfolio → backtest → metrics → research validation.

All outputs are computed from the **real market dataset** loaded at
the top of this notebook.

**Dataset SHA-256 (canonical):** `290c783af9fb803334388090c3b72df7a31f5fd580de62bb7a905250777226e7`

**Official cost model:** base scenario — 10 bps one-way per asset

**Execution model:** NEXT_CLOSE (no same-bar fills)

### 1. Research Question

Does a 3-state Gaussian HMM improve risk-adjusted returns (Sharpe ratio) for a three-asset Indian multi-asset portfolio — NIFTY 50 equity, GOLDBEES.NS gold, LIQUIDBEES.NS bond — relative to static benchmarks, under causal NEXT_CLOSE execution and realistic transaction cost assumptions?

### 2. Asset Universe

| Role | Instrument | Ticker | Notes |
|---|---|---|---|
| Equity | NIFTY 50 Index | `^NSEI` | **Index-level simulation — not directly executable** |
| Gold | Nippon India ETF Gold Bees | `GOLDBEES.NS` | INR-denominated gold ETF |
| Defensive | Nippon India Liquid Bees | `LIQUIDBEES.NS` | Liquid-bond / cash-equivalent proxy; short duration |
| Volatility feature | India VIX | `^INDIAVIX` | Optional feature only; omitted from official submitted CSV |

**Index-level tradability limitation:** `^NSEI` is an index series, not a directly tradeable instrument. Any real implementation requires a separately verified equity ETF (e.g. Nippon India NIFTY 50 BeES). This is an **index-level allocation simulation**, not an executable portfolio.

**Bond proxy:** LIQUIDBEES.NS — Nippon India Liquid Bees, INR-denominated liquid-bond / cash-equivalent proxy (NOT a sovereign bond or 10-year G-Sec; very short duration, no long-term rate hedge)

### 3. Dataset and Canonical Hash

| Property | Value |
|---|---|
| File | `data/submission_market_data.csv` |
| SHA-256 (canonical) | `290c783af9fb803334388090c3b72df7a31f5fd580de62bb7a905250777226e7` |
| First date | `2010-01-04` |
| Last date | `2026-07-27` |
| Row count | `4090` |
| Columns | `equity, gold, bond` |
| VIX included | `False` |
| Total forward-filled cells | `2` |
| Dates dropped (residual NaNs) | `0` |

Fill provenance is recorded explicitly by the data pipeline before any forward-fill occurs. Naturally constant prices (e.g. LIQUIDBEES.NS liquid-bond NAV) are not considered fill artefacts.

Third-party market data is included only for academic reproducibility; it remains subject to the terms of its original providers and exchanges.

In [1]:
from pathlib import Path
import sys, os
# Locate repo root — walk up from CWD until we find src/ or pyproject.toml
_cwd = Path.cwd()
_repo_root = _cwd
for _candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
    if (_candidate / "src").exists() or (_candidate / "pyproject.toml").exists():
        _repo_root = _candidate
        break
sys.path.insert(0, str(_repo_root / "src"))
os.chdir(str(_repo_root))  # Ensure data/ and results/ paths resolve correctly

from regime_shift.config import RegimeShiftConfig
from regime_shift.execution import ExecutionCostModel
from regime_shift.data import load_market_data_csv

config = RegimeShiftConfig()
config.execution_cost_model = ExecutionCostModel.scenario("base", config.core_assets)

DATA_PATH = "data/submission_market_data.csv"
prices = load_market_data_csv(path=DATA_PATH, config=config)
diag = prices.attrs.get("data_diagnostics", {})

import hashlib
with open(DATA_PATH, "rb") as _fh:
    _raw = _fh.read().replace(b"\r\n", b"\n")
canonical_sha256 = hashlib.sha256(_raw).hexdigest()

print(f"Price data: {len(prices)} rows")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Columns: {list(prices.columns)}")
print(f"Canonical SHA-256: {canonical_sha256}")
print(f"Forward-filled cells: {diag.get('ffill_cells_total', 0)}")
print(f"Dates dropped: {diag.get('dates_dropped', 0)}")
display(prices.head())
display(prices.describe())


Masked 2 bad print(s) in gold starting 2019-12-19 (33.6 -> 0.3355).


Price data: 4090 rows
Date range: 2010-01-04 to 2026-07-27
Columns: ['equity', 'gold', 'bond']
Canonical SHA-256: 290c783af9fb803334388090c3b72df7a31f5fd580de62bb7a905250777226e7
Forward-filled cells: 2
Dates dropped: 0


,equity,gold,bond
Date,,,
2010-01-04,5232.200195,16.549900,589.801575
2010-01-05,5277.899902,16.680000,589.801575
2010-01-06,5281.799805,16.600000,589.801575
2010-01-07,5263.100098,16.572001,589.807129
2010-01-08,5244.750000,16.490000,589.801575


,equity,gold,bond
count,4090.000000,4090.000000,4090.000000
mean,12252.154996,39.363664,760.585919
std,6534.918209,23.272216,133.808728
min,4544.200195,15.774100,589.795776
25%,6304.800049,26.067625,625.345520
50%,10324.100098,28.463115,756.482239
75%,17465.538086,44.477501,854.780701
max,26328.550781,146.529999,1000.000000


### 4. Causal NEXT_CLOSE Execution Timing

At close **t**:

1. Holdings in place before close **t** earn the **t-1 → t** return.
2. Any target signalled at close **t-1** is traded now; costs are paid at close **t**.
3. Close **t** price becomes known; features, scaler, and HMM use information strictly through **t**.
4. On rebalance dates a new regime and target are generated (the signal).
5. That signal is **not** filled at close **t**; it is filled at close **t+1**.
6. The filled target first earns the **t+1 → t+2** return.

A one-bar gap separates every signal from its fill, so no target is traded on the close that produced it. `execution_timeline.csv` records `signal_date` < `execution_date` on every fill.

```
Prices
  ↓
Causal trailing features (rolling windows, no lookahead)
  ↓
Train-only StandardScaler (fit on rolling window ≤ t)
  ↓
Rolling 3-state Gaussian HMM (fit on rolling window ≤ t)
  ↓
Regime-conditioned CVXPY allocation
  ↓
NEXT_CLOSE execution (signal at close t, fill and costs at close t+1)
  ↓
Benchmark-aligned evaluation with identical cost model
```

### 5. Leakage-Safe Features

Seven base features computed from strictly trailing windows: equity 1-day log return, 21d and 63d momentum, 21d realized volatility, 21d/63d volatility ratio, and 63d equity–gold and equity–bond correlations. Correlation is set to 0 when one leg is constant over the window (LIQUIDBEES printed a flat NAV through 2012), so no dates are dropped.
VIX features are optional and omitted when VIX is not in the dataset.

The StandardScaler is fitted only on the rolling training window (252 days through close **t**). No future data reaches the scaler.

In [2]:
from regime_shift.features import (
    compute_raw_features, drop_feature_warmup,
    fit_feature_scaler, transform_features,
)

feature_cfg = config.feature_config
raw_features = compute_raw_features(prices, config=feature_cfg)
raw_features = drop_feature_warmup(raw_features, config=feature_cfg)
print(f"Features after warmup: {len(raw_features)} rows, {len(raw_features.columns)} columns")
print(f"Feature columns: {list(raw_features.columns)}")
display(raw_features.head())


Features after warmup: 4027 rows, 7 columns
Feature columns: ['equity_log_return_1d', 'equity_momentum_21d', 'equity_momentum_63d', 'equity_volatility_21d', 'equity_volatility_ratio_21_63', 'equity_gold_correlation_63d', 'equity_bond_correlation_63d']


,equity_log_return_1d,equity_momentum_21d,equity_momentum_63d,equity_volatility_21d,equity_volatility_ratio_21_63,equity_gold_correlation_63d,equity_bond_correlation_63d
Date,,,,,,,
2010-04-08,-0.013147,0.035217,0.013809,0.109581,0.682109,0.370115,0.045790
2010-04-09,0.010744,0.051014,0.015887,0.111588,0.692486,0.366416,0.045650
2010-04-12,-0.004121,0.043675,0.010962,0.113797,0.705176,0.362208,0.045584
2010-04-13,-0.003142,0.036925,0.011372,0.115067,0.713209,0.363533,0.055894
2010-04-15,-0.009314,0.026591,0.005501,0.121256,0.747142,0.358337,0.047337


### 6. Train-Only StandardScaler and 3-Restart Gaussian HMM

Training window: **252 days** rolling.
HMM: 3 states, diagonal covariance, 3 deterministic restarts per fit.

Restart selection: highest converged training log-likelihood. **No selection based on out-of-sample performance.**

In [3]:
from regime_shift.regime_model import em_converged, em_iterations, fit_hmm

train_end = raw_features.index[min(config.train_window, len(raw_features) - 1)]
train_features = raw_features.loc[:train_end].tail(config.train_window)
scaler = fit_feature_scaler(train_features)
scaled_train = transform_features(scaler, train_features)
hmm, hidden_states, trans_mat = fit_hmm(scaled_train, train_features, config=config.hmm_config)
print(f"HMM converged (log-likelihood gain < tol, not iteration cap): {em_converged(hmm)}")
print(f"EM iterations used: {em_iterations(hmm)} of {config.hmm_config.n_iter}")
print(f"Log-likelihood: {hmm.score(scaled_train.values):.2f}")
print("Transition matrix:")
display(trans_mat)


HMM converged (log-likelihood gain < tol, not iteration cap): True
EM iterations used: 15 of 200
Log-likelihood: -1886.91
Transition matrix:


,Bull,Bear,Crisis
Bull,9.803633e-01,1.084523e-54,0.019637
Bear,1.224666e-02,9.632131e-01,0.024540
Crisis,6.862219e-43,2.536385e-02,0.974636


### 7. Bull / Bear / Crisis State Mapping

Numeric HMM states are mapped to interpretable labels using training-period volatility, momentum, and VIX statistics only. **No out-of-sample performance is used for state labelling.**

In [4]:
from regime_shift.regime_model import predict_current_state
solution = predict_current_state(
    hmm, scaler, train_features, train_features, config=config.hmm_config,
)
print(f"Current regime: {solution.regime}")
print(f"Regime probabilities: {dict(solution.probabilities)}")
print(f"Convergence: {solution.convergence}")
print(f"Iterations: {solution.n_iter}")


Current regime: Bear
Regime probabilities: {'Bull': np.float64(4.804143551207464e-22), 'Bear': np.float64(0.9997566774991676), 'Crisis': np.float64(0.00024332250089504454)}
Convergence: True
Iterations: 15


### 8. Regime-Conditioned CVXPY Portfolio Construction

Each regime has a distinct convex objective and regime-specific weight constraints:

| Regime | Objective | Key Constraints |
|---|---|---|
| Bull | Maximize return − risk_aversion × variance | Equity ≥ 45%, ≤ 80% |
| Bear | Minimize variance − return_reward × return | Equity ≤ 40%; gold+bond ≥ 60% |
| Crisis | Minimize variance | Equity ≤ 15%; gold ≥ 25%; bond ≥ 40% |

Ridge regularization is applied to the covariance matrix for PSD guarantee.
All allocations are long-only; no leverage.

In [5]:
from regime_shift.portfolio import optimize_portfolio

asset_returns = prices[["equity", "gold", "bond"]].pct_change().iloc[1:]
est_returns = asset_returns.loc[:train_end].tail(config.portfolio_config.estimation_lookback)

port_sol = optimize_portfolio(
    regime=solution.regime,
    returns_through_date=est_returns,
    previous_weights=None,
    config=config.portfolio_config,
)
print(f"Regime: {port_sol.regime}")
print(f"Solver: {port_sol.solver} | Status: {port_sol.status}")
print(f"Weights: equity={port_sol.weights['equity']:.4f}, "
      f"gold={port_sol.weights['gold']:.4f}, "
      f"bond={port_sol.weights['bond']:.4f}")


Regime: Bear
Solver: CLARABEL | Status: optimal
Weights: equity=0.2000, gold=0.8000, bond=0.0000


### 9. Transaction Costs — Base Scenario (10 bps one-way)

Cost model: **base scenario** — 10 basis points one-way per asset.

- **Initial allocation:** turnover = Σ |w_i| (full L1).
- **Subsequent rebalance:** turnover = 0.5 × Σ |w_i − w_{prev}| (half-L1).
- **Net return:** r_net = (1 + r_gross) × (1 − cost) − 1.
- No cost on non-rebalance dates.

**Limitation:** These are assumed-cost scenarios, not measured historical spreads or fills. Market impact and capacity are excluded because volume/ADV data is absent. `^NSEI` is not a directly executable fill.

Alternative scenarios: optimistic (5 bps), stressed (20 bps).

### 10. Walk-Forward Backtest with NEXT_CLOSE

- **Evaluation observations:** 3900
- **Rebalance frequency:** 21 trading days
- **Minimum training window:** 126 observations
- **Regime counts (rebalance dates):** {'Bear': 1402, 'Bull': 1385, 'Crisis': 1113}
- **Total rebalances:** 186

In [6]:
from regime_shift.backtest import run_walk_forward_backtest, run_benchmark
from regime_shift.benchmarks import static_60_40_weights, equal_weight_weights

strategy = run_walk_forward_backtest(
    prices=prices, config=config, risk_free_rate=0.0,
)
print(f"Backtest complete.")
print(f"Evaluation: {len(strategy.net_returns)} trading days")
print(f"  {strategy.net_returns.index[0].date()} to {strategy.net_returns.index[-1].date()}")
print(f"Rebalances: {strategy.rebalance_flags.sum()}")
for r in ["Bull", "Bear", "Crisis"]:
    print(f"  {r}: {strategy.regime_series.value_counts().get(r, 0)} days")


Model is not converging.  Current: -1516.9401333744677 is not greater than -1516.9397452271014. Delta is -0.00038814736626591184


Model is not converging.  Current: -1448.0770289391924 is not greater than -1448.07370786144. Delta is -0.003321077752389101


Model is not converging.  Current: -1960.0311998965042 is not greater than -1960.0311921174991. Delta is -7.779005045449594e-06


Model is not converging.  Current: -1412.2936336068371 is not greater than -1412.293613536265. Delta is -2.0070572190888925e-05


Model is not converging.  Current: -1662.928923956311 is not greater than -1662.9282969322953. Delta is -0.0006270240157846274


Model is not converging.  Current: -1674.273763312259 is not greater than -1674.2737324998502. Delta is -3.0812408795100055e-05


Model is not converging.  Current: -1663.545606073505 is not greater than -1663.5455991304816. Delta is -6.943023436178919e-06


Model is not converging.  Current: -1794.1593465409335 is not greater than -1794.159281647011. Delta is -6.489392239927838e-05


Model is not converging.  Current: -1759.5500222161584 is not greater than -1759.549621715543. Delta is -0.0004005006153420254


Model is not converging.  Current: -1738.7306439379563 is not greater than -1738.7306374012278. Delta is -6.536728506034706e-06


Backtest complete.
Evaluation: 3900 trading days
  2010-10-06 to 2026-07-27
Rebalances: 186
  Bull: 1385 days
  Bear: 1402 days
  Crisis: 1113 days


In [7]:
bench_6040 = run_benchmark(
    prices=prices, weights=static_60_40_weights(), config=config,
    rebalance_flags=strategy.rebalance_flags,
    start_date=strategy.net_returns.index[0],
)
bench_ew = run_benchmark(
    prices=prices, weights=equal_weight_weights(), config=config,
    rebalance_flags=strategy.rebalance_flags,
    start_date=strategy.net_returns.index[0],
)
print("Benchmarks complete.")


Benchmarks complete.


### 11. Official Aligned Results

All strategies share the same evaluation dates, NEXT_CLOSE execution, base 10 bps cost model, and risk-free rate of 0.

In [8]:
from regime_shift.metrics import compute_performance_metrics
import pandas as pd

def _row(label, net, gross, turn, costs):
    m = compute_performance_metrics(
        net, gross_returns=gross, turnover=turn, transaction_costs=costs, risk_free_rate=0.0)
    d = m.to_dict(); d["Strategy"] = label; return d

perf_rows = [
    _row("RegimeShift Gross", strategy.gross_returns, strategy.gross_returns,
         strategy.turnover, strategy.transaction_costs),
    _row("RegimeShift Net",   strategy.net_returns,   strategy.gross_returns,
         strategy.turnover, strategy.transaction_costs),
    _row("Static 60/40 Gross", bench_6040.gross_returns, bench_6040.gross_returns,
         bench_6040.turnover, bench_6040.transaction_costs),
    _row("Static 60/40 Net",   bench_6040.net_returns,   bench_6040.gross_returns,
         bench_6040.turnover, bench_6040.transaction_costs),
    _row("Equal Weight Gross", bench_ew.gross_returns, bench_ew.gross_returns,
         bench_ew.turnover, bench_ew.transaction_costs),
    _row("Equal Weight Net",   bench_ew.net_returns,   bench_ew.gross_returns,
         bench_ew.turnover, bench_ew.transaction_costs),
]
perf_df = pd.DataFrame(perf_rows)
display(perf_df[["Strategy","CAGR","Sharpe","Sortino","Maximum Drawdown",
                  "Calmar","Annualised Turnover","Transaction Cost Drag"]])


,Strategy,CAGR,Sharpe,Sortino,Maximum Drawdown,Calmar,Annualised Turnover,Transaction Cost Drag
0,RegimeShift Gross,0.0816,0.8819,0.8644,0.1447,0.5639,3.2165,0.0000
1,RegimeShift Net,0.0747,0.8145,0.7983,0.1447,0.5163,3.2165,0.3161
2,Static 60/40 Gross,0.0718,0.7583,0.7297,0.2338,0.3071,0.1692,0.0000
3,Static 60/40 Net,0.0715,0.7555,0.7273,0.2339,0.3058,0.1692,0.0124
4,Equal Weight Gross,0.0895,1.1833,1.1508,0.1495,0.5988,0.2012,0.0000
5,Equal Weight Net,0.0892,1.1788,1.1469,0.1495,0.5963,0.2012,0.0197


### 12. Ablation Study Results

| strategy | sharpe | cagr | max_drawdown | ann_turnover |
|---|---|---|---|---|
| A_RegimeShift_Full | 0.8145 | 0.0747 | 0.1447 | 3.2165 |
| B_NoRegime_Optimizer | 0.7666 | 0.0861 | 0.2635 | 0.9758 |
| C_VolRule | 0.8438 | 0.0844 | 0.2557 | 0.4497 |
| D_HMM_Fixed | 0.8921 | 0.0673 | 0.1524 | 1.7024 |
| E_MinVariance | 2.1145 | 0.0503 | 0.0440 | 0.0868 |
| F_Static_6040 | 0.7555 | 0.0715 | 0.2339 | 0.1692 |
| G_EqualWeight | 1.1788 | 0.0892 | 0.1495 | 0.2012 |

Ablation strategies: (A) Full RegimeShift HMM+CVXPY, (B) No-regime optimizer (always Bull), (C) Volatility-rule allocation, (D) HMM+fixed weights, (E) Minimum variance, (F) Static 60/40, (G) Equal Weight. All share identical dates, costs, and execution.

### 13. Chronological Subperiod Analysis

> **Important:** The retrospective evaluation period (2022-2026) is **not** a pristine holdout because the full historical sample was inspected during earlier project development.

| period | n_obs | cagr | sharpe | max_drawdown |
|---|---|---|---|---|
| Development | 2028 | 0.0456 | 0.5635 | 0.1282 |
| Validation | 742 | 0.1224 | 1.1186 | 0.1447 |
| Retrospective Evaluation | 1130 | 0.0974 | 0.9993 | 0.1035 |


### 14. Rolling-Origin Evaluation (Calendar Folds)

Each fold is a calendar slice of the single walk-forward run (the model is already refit every 21 days on a trailing window), not an independent re-fit per fold.

| fold | n_obs | cagr | sharpe | max_drawdown |
|---|---|---|---|---|
| 2013-2015 | 737 | 0.0249 | 0.3602 | 0.1282 |
| 2016-2018 | 740 | 0.0275 | 0.4529 | 0.1162 |
| 2019-2021 | 742 | 0.1224 | 1.1186 | 0.1447 |
| 2022-2024 | 739 | 0.0412 | 0.5358 | 0.0976 |
| 2025-2026 | 391 | 0.2119 | 1.6306 | 0.1035 |


### 15. Bootstrap Uncertainty (Paired Moving-Block, n=2000, block=21)

| metric | mean_diff | ci_lower_95 | ci_upper_95 | zero_inside_ci | statistically_significant | n_valid | note |
|---|---|---|---|---|---|---|---|
| RegimeShift_sharpe_vs_6040 | 0.0491 | -0.4518 | 0.5561 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_sharpe_vs_EW | -0.3737 | -0.7162 | -0.0210 | False | True | 2000 | Zero outside CI; causal retrospective data only |
| RegimeShift_cagr_vs_6040 | 0.0024 | -0.0500 | 0.0566 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_cagr_vs_EW | -0.0145 | -0.0437 | 0.0188 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_max_drawdown_vs_6040 | -0.0541 | -0.1908 | 0.0664 | True | False | 2000 | No significance claimed when zero lies inside CI |
| RegimeShift_max_drawdown_vs_EW | 0.0363 | -0.0298 | 0.1320 | True | False | 2000 | No significance claimed when zero lies inside CI |

No significance is claimed when zero lies inside a confidence interval. All results are retrospective causal simulation data only.

### 16. HMM Stability Diagnostics

- Total rolling HMM fits: **186**
- Convergence rate: **100.0%**
- Median log-likelihood: **-1802.64**
- State occupancy (Bear): **36.0%**
- State occupancy (Bull): **35.5%**
- State occupancy (Crisis): **28.5%**


### 17. Findings and Negative Findings

Under NEXT_CLOSE timing (signal at close t, fill at close t+1) and base 10 bps costs:

| Strategy | CAGR | Sharpe | Max Drawdown |
|---|---|---|---|
| RegimeShift | 7.47% | 0.815 | 14.47% |
| Static 60/40 | 7.15% | 0.755 | 23.39% |
| Equal Weight | 8.92% | 1.179 | 14.95% |

(Same numbers as `results/submission/performance_summary.csv`.)

**What this research demonstrated:**
- Versus static 60/40, RegimeShift has Sharpe 0.81 vs 0.76 and max drawdown 14.5% vs 23.4%; the bootstrap CI for the Sharpe difference includes zero, so the edge is not significant.
- Equal weight (Sharpe 1.18) beats RegimeShift; the paired bootstrap places that Sharpe gap outside zero.
- The ablation attributes the drawdown reduction to the HMM regime signal; the CVXPY layer adds turnover without beating fixed per-regime weights.
- Turnover of 3.2× a year (vs 0.17× for 60/40) creates the main cost drag.

### 18. Limitations

1. **Index-level equity simulation:** `^NSEI` is an index, not a directly executable instrument. A real portfolio requires an equity ETF with verified common history, corporate-action treatment, and liquidity audit.
2. **Assumed costs, not measured:** 5/10/20 bps are scenario assumptions. Market impact, bid-ask spreads, and capacity are not modelled.
3. **Short-duration bond proxy:** LIQUIDBEES.NS provides no long-term duration hedge or inflation protection.
4. **Retrospective evaluation:** All periods were visible during development. No segment is a pristine holdout.
5. **Three assets only:** Broader diversification requires expanding the universe.
6. **Gaussian HMM:** Student-t emissions may better capture fat tails.
7. **No alpha proven:** Positive CAGR reflects the underlying market uptrend; the Sharpe edge over 60/40 is not statistically significant.
8. **Risk-free rate is 0:** Sharpe ratios of cash-heavy portfolios are overstated, and the LIQUIDBEES `Close` series omits most of its distributions.

### 19. Reproduction

```bash
pip install -e ".[dev]"

# Official experiment
python run_submission.py \
    --data-path data/submission_market_data.csv \
    --cost-scenario base \
    --output-dir results/submission

# Research validation suite
python scripts/run_research_suite.py

# Rebuild and execute this notebook
python scripts/build_notebook.py
python -m jupyter nbconvert --to notebook --execute \
    notebooks/RegimeShift_Submission.ipynb --inplace \
    --ExecutePreprocessor.timeout=3600

# Tests
python -m pytest -q
python scripts/verify_release.py
```